In [1]:
# @title LICENSE
# Licensed under the Apache License, Version 2.0

## JaxPruner Quick Start
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/jaxpruner/blob/main/colabs/quick_start.ipynb)

This interactive colab provides a short overview of some of the key features of the `jaxpruner` library:

- One-shot Pruning
- Pruning during Optimization (Integration w/ optax)
- ConfigDict Integration
- Compatibility with JAX parallelization via `pmap` and sharded `jax.jit`

In [2]:
import functools
import flax
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec
import numpy as np
import optax
import pprint

In [3]:
import sys
if 'google.colab' in sys.modules:
  !pip3 install git+https://github.com/dimarkov/jaxpruner
import jaxpruner
import ml_collections


# One-shot Pruning
Pruning a given matrix to a desired level of sparsity is the building block of any pruning algorithm. Therefore jaxpruner provides a common API for one-shot
pruning. This is achieved by calling the `instant_sparsify` method.

In [4]:
matrix_size = 5
learning_rate = 0.01
matrix = jax.random.uniform(
    jax.random.PRNGKey(8), shape=(matrix_size, matrix_size)
)
print(matrix)

[[0.29064357 0.08548212 0.9970648  0.71362007 0.15596008]
 [0.12801719 0.20627809 0.9185562  0.03574014 0.17216587]
 [0.7206156  0.44509602 0.61574686 0.6491784  0.02918768]
 [0.7946211  0.1640774  0.07161057 0.7137177  0.71402645]
 [0.4283384  0.02327716 0.59752345 0.66846883 0.7168479 ]]


In [5]:
sparsity_distribution = functools.partial(
    jaxpruner.sparsity_distributions.uniform, sparsity=0.8
)
pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=sparsity_distribution
)
pruned_matrix, mask = pruner.instant_sparsify(matrix)

print(pruned_matrix)
print(mask.dtype)
print(mask)

[[0.        0.        0.9970648 0.        0.       ]
 [0.        0.        0.9185562 0.        0.       ]
 [0.7206156 0.        0.        0.        0.       ]
 [0.7946211 0.        0.        0.        0.       ]
 [0.        0.        0.        0.        0.7168479]]
uint8
[[0 0 1 0 0]
 [0 0 1 0 0]
 [1 0 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 1]]


We can quickly change the sparsity structure using `sparsity_type` flag.

In [6]:
pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=sparsity_distribution,
    sparsity_type=jaxpruner.sparsity_types.NByM(1, 5),
)
pruned_matrix, mask = pruner.instant_sparsify(matrix)

print(pruned_matrix)
print(mask.dtype)
print(mask)

[[0.        0.        0.9970648 0.        0.       ]
 [0.        0.        0.9185562 0.        0.       ]
 [0.7206156 0.        0.        0.        0.       ]
 [0.7946211 0.        0.        0.        0.       ]
 [0.        0.        0.        0.        0.7168479]]
uint8
[[0 0 1 0 0]
 [0 0 1 0 0]
 [1 0 0 0 0]
 [1 0 0 0 0]
 [0 0 0 0 1]]


`instant sparsify` also supports parameter collections, which are commonly used in deep learning.

In [7]:
# params = [matrix, 1 - matrix]
params = {'pos': matrix, 'inv': 1 - matrix}
pruned_params, masks = pruner.instant_sparsify(params)
pprint.pprint(pruned_params)

{'inv': Array([[0.        , 0.9145179 , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.96425986, 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.9708123 ],
       [0.        , 0.        , 0.92838943, 0.        , 0.        ],
       [0.        , 0.97672284, 0.        , 0.        , 0.        ]],      dtype=float32),
 'pos': Array([[0.       , 0.       , 0.9970648, 0.       , 0.       ],
       [0.       , 0.       , 0.9185562, 0.       , 0.       ],
       [0.7206156, 0.       , 0.       , 0.       , 0.       ],
       [0.7946211, 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.7168479]],      dtype=float32)}


It is common to choose different sparsities for different layers or keep them dense entirely. We provide some basic functions to distribute sparsity across different layers such as `uniform` (default) and `erk` under `jaxpruner.sparsity_distributions`. Users can also define their own distributions easily. Here we define a custom distribution function to set different sparsities for each variable.

In [8]:
def custom_distribution(params, sparsity=0.8):
  return {key: 0.4 if 'pos' in key else sparsity for key in params}


pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=custom_distribution
)
pruned_params, masks = pruner.instant_sparsify(params)
pprint.pprint(jaxpruner.summarize_sparsity(pruned_params))

{'_nparams': Array(50., dtype=float32),
 '_nparams_nnz': Array(20., dtype=float32),
 '_total_sparsity': Array(0.6, dtype=float32),
 'inv': Array(0.8, dtype=float32),
 'pos': Array(0.40000004, dtype=float32)}


Masks used for enforcing sparsity use the same tree structure as the parameters pruned. We use `None` values to indicate dense parameters. We don't create masks for dense variables.

In [9]:
def custom_distribution2(params, sparsity=0.8):
  return {key: None if 'pos' in key else sparsity for key in params}


pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=custom_distribution2
)
_, masks = pruner.instant_sparsify(params)
pprint.pprint(masks)

{'inv': Array([[0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1],
       [0, 0, 1, 0, 0],
       [0, 1, 0, 0, 0]], dtype=uint8),
 'pos': None}


Changing the pruning algorithm is easy as they all inherit from the same `BaseUpdater`. We have the following baseline pruning and sparse training algorithms included in our library.

In [10]:
for k in jaxpruner.ALGORITHM_REGISTRY:
  print(k, jaxpruner.ALGORITHM_REGISTRY[k])

no_prune <class 'jaxpruner.base_updater.NoPruning'>
magnitude <class 'jaxpruner.algorithms.pruners.MagnitudePruning'>
random <class 'jaxpruner.algorithms.pruners.RandomPruning'>
saliency <class 'jaxpruner.algorithms.pruners.SaliencyPruning'>
magnitude_ste <class 'jaxpruner.algorithms.ste.SteMagnitudePruning'>
random_ste <class 'jaxpruner.algorithms.ste.SteRandomPruning'>
global_magnitude <class 'jaxpruner.algorithms.global_pruners.GlobalMagnitudePruning'>
global_saliency <class 'jaxpruner.algorithms.global_pruners.GlobalSaliencyPruning'>
static_sparse <class 'jaxpruner.algorithms.sparse_trainers.StaticRandomSparse'>
rigl <class 'jaxpruner.algorithms.sparse_trainers.RigL'>
set <class 'jaxpruner.algorithms.sparse_trainers.SET'>


Next we use gradient based saliency score for pruning. `SaliencyPruning` requires gradients to be passed to `pruner.instant_sparsify`. Gradients are multipled with parameter values to obtain a first order Taylor approximation of the change in loss.

In [11]:
# Gradient based pruning
pruner = jaxpruner.SaliencyPruning(
    sparsity_distribution_fn=sparsity_distribution
)
print(pruner.instant_sparsify(matrix, grads=(1 - matrix))[0])

[[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.44509602 0.61574686 0.6491784  0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.4283384  0.         0.59752345 0.         0.        ]]


# Pruning as optimization (jaxpruner + optax)

Often state-of-the-art pruning algorithms require iterative adjustments to the sparsity masks used. Such iterative approaches are stateful, i.e. they require some additional variables like masks, counters and initial values. This is similar to common optimization algorithms such as Adam and SGD+Momentum which require moving averages.

The observation that *most iterative pruning and sparse training algoritms can be implemented as an optimizer*, played a key role when designing `jaxpruner` and led us to integrate `jaxpruner` with the `optax` optimization library.

Here is an example training loop where we find an orthogonal matrix using gradient descent:

In [12]:
matrix_size = 5


def loss_fn(params):
  matrix = params['w']
  loss = jnp.sum((matrix @ matrix.T - jnp.eye(matrix_size)) ** 2)
  return loss


grad_fn = jax.value_and_grad(loss_fn)


@functools.partial(jax.jit, static_argnames='optimizer')
def update_fn(params, opt_state, optimizer):
  loss, grads = grad_fn(params)
  updates, opt_state = optimizer.update(grads, opt_state, params)
  params = optax.apply_updates(params, updates)
  return params, opt_state, loss


def run_experiment(init_matrix):
  optimizer = optax.sgd(0.05)
  params = {'w': init_matrix}
  opt_state = optimizer.init(params)

  for i in range(20):
    params, opt_state, loss = update_fn(params, opt_state, optimizer)
    if i % 4 == 0:
      print(f'Step: {i}, loss: {loss}')
  return params['w']

First run the baseline training with a dense matrix.

In [13]:
params = jax.random.uniform(
    jax.random.PRNGKey(8), shape=(matrix_size, matrix_size)
)
run_experiment(params)

Step: 0, loss: 23.943775177001953
Step: 4, loss: 2.2471885681152344
Step: 8, loss: 1.5628561973571777
Step: 12, loss: 1.034632682800293
Step: 16, loss: 0.6822665333747864


Array([[-0.00625665, -0.43513545,  0.5891121 ,  0.5650556 , -0.2578325 ],
       [ 0.13793057,  0.20787714,  0.7358833 , -0.56433344,  0.24054722],
       [ 0.43346745,  0.69476914,  0.1465539 ,  0.345813  , -0.34895536],
       [ 0.74850273, -0.13608426, -0.20598379,  0.15530662,  0.47220108],
       [-0.256776  ,  0.10510055,  0.18718539,  0.4198591 ,  0.71483105]],      dtype=float32)

Adding a pruner to an existing training loop requires just 2 lines. First we wrap an existing optimizer using the `pruner.wrap_optax` method. This wrapped optimizer ensures the masks are updated during the training. Second, we add a `pruner.post_gradient_update` call after our gradient step. This function defines algorithm specific parameter updates (like applying a mask to parameters) and provides flexibility when implementing various algorithms.

In [14]:
def run_pruning_experiment(init_matrix, pruner):
  optimizer = optax.sgd(0.05)
  # Modification #1
  optimizer = pruner.wrap_optax(optimizer)

  params = {'w': init_matrix}
  opt_state = optimizer.init(params)

  for i in range(20):
    params, opt_state, loss = update_fn(params, opt_state, optimizer)
    # Modification #2
    params = pruner.post_gradient_update(params, opt_state)

    if i % 4 == 0:
      print(f'Step: {i}, loss: {loss}')
      print(jaxpruner.summarize_sparsity(params, only_total_sparsity=True))
  return params['w']

Now, prune the matrix in one step (step=15).




In [15]:
pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=sparsity_distribution,
    scheduler=jaxpruner.sparsity_schedules.OneShotSchedule(target_step=10),
)
params = jax.random.uniform(
    jax.random.PRNGKey(8), shape=(matrix_size, matrix_size)
)
run_pruning_experiment(params, pruner)

Step: 0, loss: 23.943775177001953
{'_total_sparsity': Array(0., dtype=float32), '_nparams_nnz': Array(25., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 4, loss: 2.2471885681152344
{'_total_sparsity': Array(0., dtype=float32), '_nparams_nnz': Array(25., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 8, loss: 1.5628561973571777
{'_total_sparsity': Array(0., dtype=float32), '_nparams_nnz': Array(25., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 12, loss: 2.3711094856262207
{'_total_sparsity': Array(0.8, dtype=float32), '_nparams_nnz': Array(5., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 16, loss: 1.5318175554275513
{'_total_sparsity': Array(0.8, dtype=float32), '_nparams_nnz': Array(5., dtype=float32), '_nparams': Array(25., dtype=float32)}


Array([[-0.        , -0.        ,  0.        ,  0.873682  , -0.        ],
       [ 0.        ,  0.        ,  0.8961111 , -0.29732317,  0.        ],
       [ 0.        ,  0.97839886,  0.        ,  0.        , -0.        ],
       [ 0.        ,  0.        , -0.        ,  0.        ,  0.        ],
       [-0.        , -0.        ,  0.        ,  0.        ,  0.98656356]],      dtype=float32)

Alternatively we can prune it iteratively using the [polynomial schedule](https://arxiv.org/abs/1710.01878).

In [16]:
pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=sparsity_distribution,
    scheduler=jaxpruner.sparsity_schedules.PolynomialSchedule(
        update_freq=4, update_start_step=2, update_end_step=14
    ),
)
params = jax.random.uniform(
    jax.random.PRNGKey(8), shape=(matrix_size, matrix_size)
)
run_pruning_experiment(params, pruner)

Step: 0, loss: 23.943775177001953
{'_total_sparsity': Array(0., dtype=float32), '_nparams_nnz': Array(25., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 4, loss: 2.2471885681152344
{'_total_sparsity': Array(0.48000002, dtype=float32), '_nparams_nnz': Array(13., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 8, loss: 1.7219597101211548
{'_total_sparsity': Array(0.76, dtype=float32), '_nparams_nnz': Array(6., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 12, loss: 2.4149537086486816
{'_total_sparsity': Array(0.8, dtype=float32), '_nparams_nnz': Array(5., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 16, loss: 2.0261309146881104
{'_total_sparsity': Array(0.8, dtype=float32), '_nparams_nnz': Array(5., dtype=float32), '_nparams': Array(25., dtype=float32)}


Array([[-0.       ,  0.       ,  0.       ,  0.       , -0.       ],
       [ 0.       ,  0.       ,  0.712454 , -0.7011148,  0.       ],
       [ 0.6729078,  0.7316286,  0.       ,  0.       , -0.       ],
       [ 0.       , -0.       , -0.       ,  0.       ,  0.       ],
       [ 0.       ,  0.       ,  0.       ,  0.       ,  0.9890534]],      dtype=float32)

# ml_collections.ConfigDict Integration

Many popular jax libraries like [scenic](https://github.com/google-research/scenic) and [big_vision](https://github.com/google-research/big_vision) use `ml_collections.ConfigDict` to configure experiments. `jaxpruner` provides a helper function (`jaxpruner.create_updater_from_config`) to make it easy to use a `ConfigDict` to generate pruner objects.

In [17]:
sparsity_config = ml_collections.ConfigDict()
sparsity_config.algorithm = 'magnitude'
sparsity_config.update_freq = 2
sparsity_config.update_end_step = 15
sparsity_config.update_start_step = 5
sparsity_config.sparsity = 0.6
sparsity_config.dist_type = 'uniform'

In [18]:
# Create a dense layer and sparsify.
pruner = jaxpruner.create_updater_from_config(sparsity_config)
params = jax.random.uniform(
    jax.random.PRNGKey(8), shape=(matrix_size, matrix_size)
)
run_pruning_experiment(params, pruner)

Step: 0, loss: 23.943775177001953
{'_total_sparsity': Array(0., dtype=float32), '_nparams_nnz': Array(25., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 4, loss: 2.2471885681152344
{'_total_sparsity': Array(0., dtype=float32), '_nparams_nnz': Array(25., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 8, loss: 1.7635594606399536
{'_total_sparsity': Array(0.48000002, dtype=float32), '_nparams_nnz': Array(13., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 12, loss: 1.1568949222564697
{'_total_sparsity': Array(0.6, dtype=float32), '_nparams_nnz': Array(10., dtype=float32), '_nparams': Array(25., dtype=float32)}
Step: 16, loss: 0.8065167665481567
{'_total_sparsity': Array(0.6, dtype=float32), '_nparams_nnz': Array(10., dtype=float32), '_nparams': Array(25., dtype=float32)}


Array([[ 0.        , -0.        ,  0.6641318 ,  0.7387717 ,  0.        ],
       [ 0.        , -0.        ,  0.74121106, -0.66904   ,  0.        ],
       [ 0.46330404,  0.7176696 ,  0.        ,  0.        , -0.36254412],
       [ 0.78217924, -0.        , -0.        , -0.        ,  0.42377493],
       [-0.        ,  0.        ,  0.        ,  0.        ,  0.7663318 ]],      dtype=float32)

# Parallelization with `pmap` and sharded `jax.jit`

The `jaxpruner` library is in general compatible with JAX parallelization mechanisms like `pmap` and `jax.jit` with shardings (the former `pjit`, which was merged into `jax.jit`). There are some minor points to watch out for,
which we will now demonstrate using parallelized versions of the previously introduced orthogonal matrix optimization example.

## `pmap`

First, we demonstrate compatibility with `pmap` where a model is replicated to run different shards of a batch on different devices. Note that this example
has no actual model "inputs" apart from the parameter matrix and the replication is thus not directly useful, but the general mechanisms are the same as for real training.

The main point to watch out for is to make sure that the optimizer state is replicated **after** wrapping it with the `jaxpruner`.

In [19]:
matrix_size = 8


def loss_fn(params):
  matrix = params['w']
  loss = jnp.sum((matrix @ matrix.T - jnp.eye(matrix_size)) ** 2)
  return loss


grad_fn = jax.value_and_grad(loss_fn)


@functools.partial(
    jax.pmap,
    out_axes=(0, 0, None),
    axis_name='batch',
    static_broadcasted_argnums=(2,),
)
def update_fn(params, opt_state, optimizer):
  loss, grads = grad_fn(params)
  loss = jax.lax.pmean(loss, 'batch')
  grads = jax.lax.pmean(grads, 'batch')
  updates, opt_state = optimizer.update(grads, opt_state, params)
  params = optax.apply_updates(params, updates)
  return params, opt_state, loss


sparsity_distribution = functools.partial(
    jaxpruner.sparsity_distributions.uniform, sparsity=0.8
)

pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=sparsity_distribution,
    scheduler=jaxpruner.sparsity_schedules.OneShotSchedule(target_step=0),
)

optimizer = optax.sgd(0.001)
optimizer = pruner.wrap_optax(optimizer)
params = {
    'w': jax.random.normal(jax.random.PRNGKey(0), (matrix_size, matrix_size))
}
opt_state = optimizer.init(params)
# The key step for using pmap with the jaxpruner is to replicate the optimizer
# state **after** wrapping it.
opt_state = flax.jax_utils.replicate(opt_state)
params = flax.jax_utils.replicate(params)

for i in range(100):
  params, opt_state, loss = update_fn(params, opt_state, optimizer)
  params = pruner.post_gradient_update(params, opt_state)
  if i % 5 == 0:
    print(f'Step: {i}, loss: {loss}')
params = flax.jax_utils.unreplicate(params)
print(params['w'])

Step: 0, loss: 833.542236328125
Step: 5, loss: 135.766845703125
Step: 10, loss: 81.42095947265625
Step: 15, loss: 54.50904846191406
Step: 20, loss: 39.08252716064453
Step: 25, loss: 29.462478637695312
Step: 30, loss: 23.02825927734375
Step: 35, loss: 18.535564422607422
Step: 40, loss: 15.283319473266602
Step: 45, loss: 12.864278793334961
Step: 50, loss: 11.006962776184082
Step: 55, loss: 9.577396392822266
Step: 60, loss: 8.437570571899414
Step: 65, loss: 7.525918006896973
Step: 70, loss: 6.785754680633545
Step: 75, loss: 6.175198078155518
Step: 80, loss: 5.675952911376953
Step: 85, loss: 5.253851890563965
Step: 90, loss: 4.899274826049805
Step: 95, loss: 4.603663444519043
[[ 0.6220028   1.042128   -0.         -0.          0.         -0.
  -0.          0.        ]
 [ 0.         -0.          0.9844511  -0.6136205   0.         -0.
   0.          0.32181045]
 [-0.          0.         -0.         -0.6186563  -0.63202316  0.79884535
   0.         -0.        ]
 [ 0.          0.          0.   

## Sharded `jax.jit`

Next, we demonstrate tensor sharded training with `jax.jit` and its `in_shardings` and `out_shardings` arguments. Here the key is that the partition specifications of the wrapped optimizer state have to incoporate also the `jaxpruner.base_update.SparseState` produced by the pruning wrapper.

In [20]:
matrix_size = 8
if jax.device_count() % 8 == 0:
  MESH_SHAPE = (2, 4)
else:
  MESH_SHAPE = (1, 1)


def loss_fn(params):
  matrix = params['w']
  loss = jnp.sum((matrix @ matrix.T - jnp.eye(matrix_size)) ** 2)
  return loss


grad_fn = jax.value_and_grad(loss_fn)

# Define the partition-specs for the sharded jax.jit; in most libraries for
# real models this is done somewhat automatically, yet this will likely require
# a small adjustment as shown below.

params_partition = {'w': PartitionSpec('X', 'Y')}

# The main step required to run the jaxpruner together with a sharded jax.jit
# is defining a partition-spec for the wrapped `SparseState` as shown below.
opt_partition = jaxpruner.base_updater.SparseState(
    masks=params_partition,
    inner_state=(None, None),  # other optimizers may require sharding
    target_sparsities=None,
    count=None,
)

resources = (params_partition, opt_partition)


@functools.partial(
    jax.jit,
    in_shardings=resources,
    out_shardings=resources + (None,),
    static_argnames='optimizer',
)
def update_fn(params, opt_state, optimizer):
  loss, grads = grad_fn(params)
  updates, opt_state = optimizer.update(grads, opt_state, params)
  params = optax.apply_updates(params, updates)
  return params, opt_state, loss


sparsity_distribution = functools.partial(
    jaxpruner.sparsity_distributions.uniform, sparsity=0.8
)
pruner = jaxpruner.MagnitudePruning(
    sparsity_distribution_fn=sparsity_distribution,
    scheduler=jaxpruner.sparsity_schedules.OneShotSchedule(target_step=0),
)

optimizer = optax.sgd(0.001)
optimizer = pruner.wrap_optax(optimizer)
params = {
    'w': jax.random.normal(jax.random.PRNGKey(0), (matrix_size, matrix_size))
}
opt_state = optimizer.init(params)

devices = np.asarray(jax.devices()[: int(np.prod(MESH_SHAPE))]).reshape(
    MESH_SHAPE
)
mesh = jax.sharding.Mesh(devices, ('X', 'Y'))

with jax.set_mesh(mesh):
  for i in range(100):
    params, opt_state, loss = update_fn(params, opt_state, optimizer)
    params = pruner.post_gradient_update(params, opt_state)
    if i % 5 == 0:
      print(f'Step: {i}, loss: {loss}')
  print(params['w'])
  jax.debug.visualize_array_sharding(params['w'])

Step: 0, loss: 833.542236328125
Step: 5, loss: 135.766845703125
Step: 10, loss: 81.42095947265625
Step: 15, loss: 54.50904846191406
Step: 20, loss: 39.08252716064453
Step: 25, loss: 29.462478637695312
Step: 30, loss: 23.02825927734375
Step: 35, loss: 18.535564422607422
Step: 40, loss: 15.283319473266602
Step: 45, loss: 12.864278793334961
Step: 50, loss: 11.006962776184082
Step: 55, loss: 9.577396392822266
Step: 60, loss: 8.437570571899414
Step: 65, loss: 7.525918006896973
Step: 70, loss: 6.785754680633545
Step: 75, loss: 6.175198078155518
Step: 80, loss: 5.675952911376953
Step: 85, loss: 5.253851890563965
Step: 90, loss: 4.899274826049805
Step: 95, loss: 4.603663444519043
[[ 0.6220028   1.042128   -0.         -0.          0.         -0.
  -0.          0.        ]
 [ 0.         -0.          0.9844511  -0.6136205   0.         -0.
   0.          0.32181045]
 [-0.          0.         -0.         -0.6186563  -0.63202316  0.79884535
   0.         -0.        ]
 [ 0.          0.          0.   

                         
                         
                         
                         
                         
          GPU 0          
                         
                         
                         
                         
                         